# 01 – Data Collection (Python → CSV → MySQL)

**Goal:** pull AAPL price history and financial statements with `yfinance`, check that the data is consistent, then save it to CSV and MySQL.

**Steps**
1. Download daily prices for AAPL, 3 peers (MSFT, GOOGL, META) and the S&P 500 (`^GSPC`)
2. Download AAPL's Income Statement, Balance Sheet and Cash Flow
3. Keep only the columns needed for the analysis
4. Check the data with two accounting rules (Assets = Liabilities + Equity, FCF = CFO + Capex)
5. Save to CSV and push to MySQL (`aapl_stock_project` schema)

> Note: `yfinance` only gives about 4–5 years of annual statements. The cleaned 10-K figures used in the Excel model are saved in `../data/` so the project can be re-run without the internet.

In [1]:
import os
from pathlib import Path
import pandas as pd

REFRESH_DATA = False  # Set True only to intentionally update the market snapshot.
DATA_AS_OF = "2026-11-13"
main_ticker = "AAPL"
peer_ticker = ["MSFT", "GOOGL", "META"]
benchmark = "^GSPC"
all_ticker = [main_ticker] + peer_ticker + [benchmark]
print(all_ticker)


['AAPL', 'MSFT', 'GOOGL', 'META', '^GSPC']


## 1. Price data

In [2]:
if REFRESH_DATA:
    import yfinance as yf
    price_data = yf.download(all_ticker, start="2015-01-01", group_by="ticker", auto_adjust=True)
    print("Live download:", price_data.shape)
else:
    price_long = pd.read_csv("../data/raw/price_data_long.csv", parse_dates=["date"])
    print("Saved snapshot:", DATA_AS_OF, price_long.shape)
    assert price_long["date"].max() == pd.Timestamp(DATA_AS_OF)


Saved snapshot: 2026-11-13 (14925, 7)


The table has **2 levels of columns** (ticker → Open/High/Low/Close/Volume). SQL cannot store this shape well, so we reshape it to **long format**: one row = one ticker on one day.

In [3]:
if REFRESH_DATA:
    price_long = price_data.stack(level=0).reset_index()
    price_long.columns = [c.lower() for c in price_long.columns]
    price_long = price_long.dropna(subset=["close"])
assert not price_long.duplicated(["date", "ticker"]).any()
print(price_long.shape)
price_long.head()


(14925, 7)


,date,ticker,open,high,low,close,volume
0,2015-01-02,^GSPC,2058.899902,2072.360107,2046.040039,2058.199951,2708700000
1,2015-01-02,GOOGL,26.378985,26.537479,26.145210,26.227924,26480000
2,2015-01-02,META,77.834529,78.181207,76.962872,77.705757,18177500
3,2015-01-02,AAPL,24.608505,24.619552,23.715980,24.153406,212818400
4,2015-01-02,MSFT,39.522487,40.166230,39.420844,39.607189,27913900


## 2. Financial statements

In [4]:
if REFRESH_DATA:
    aapl = yf.Ticker("AAPL")
    income_statement = aapl.financials.T.sort_index()
    balance_sheet = aapl.balance_sheet.T.sort_index()
    cash_flow = aapl.cashflow.T.sort_index()
else:
    income_statement = pd.read_csv("../data/raw/yf_income_statement.csv", index_col=0, parse_dates=True)
    balance_sheet = pd.read_csv("../data/raw/yf_balance_sheet.csv", index_col=0, parse_dates=True)
    cash_flow = pd.read_csv("../data/raw/yf_cash_flow.csv", index_col=0, parse_dates=True)
income_statement.index


DatetimeIndex(['2023-09-30', '2024-09-30', '2025-09-30', '2026-09-30'], dtype='datetime64[us]', freq=None)

In [5]:
key_columns_is = ["Total Revenue", "Cost Of Revenue", "Gross Profit", "Research And Development",
                  "Selling General And Administration", "Operating Income", "EBITDA",
                  "Pretax Income", "Tax Provision", "Net Income", "Diluted EPS", "Diluted Average Shares"]

key_columns_bs = ["Total Assets", "Current Assets", "Current Liabilities", "Cash And Cash Equivalents",
                  "Inventory", "Accounts Receivable", "Total Liabilities Net Minority Interest",
                  "Total Debt", "Stockholders Equity"]

key_columns_cf = ["Operating Cash Flow", "Capital Expenditure", "Free Cash Flow",
                  "Depreciation And Amortization", "Cash Dividends Paid", "Repurchase Of Capital Stock"]

# reindex = keep these columns (a missing column becomes NaN instead of a KeyError)
income_statement_clean = income_statement.reindex(columns=key_columns_is)
balance_sheet_clean = balance_sheet.reindex(columns=key_columns_bs)
cash_flow_clean = cash_flow.reindex(columns=key_columns_cf)

# drop years where every value is missing (the oldest year is often empty in yfinance)
income_statement_clean = income_statement_clean.dropna(how="all")
balance_sheet_clean = balance_sheet_clean.dropna(how="all")
cash_flow_clean = cash_flow_clean.dropna(how="all")

income_statement_clean

,Total Revenue,Cost Of Revenue,Gross Profit,Research And Development,Selling General And Administration,Operating Income,EBITDA,Pretax Income,Tax Provision,Net Income,Diluted EPS,Diluted Average Shares
2023-09-30,3.832850e+11,2.141370e+11,1.691480e+11,2.991500e+10,2.493200e+10,1.143010e+11,1.258200e+11,1.137360e+11,1.674100e+10,9.699500e+10,6.13,1.581255e+10
2024-09-30,3.910350e+11,2.103520e+11,1.806830e+11,3.137000e+10,2.609700e+10,1.232160e+11,1.346610e+11,1.234850e+11,2.974900e+10,9.373600e+10,6.08,1.540810e+10
2025-09-30,4.161610e+11,2.209600e+11,1.952010e+11,3.455000e+10,2.760100e+10,1.330500e+11,1.447480e+11,1.327290e+11,2.071900e+10,1.120100e+11,7.46,1.500470e+10
2026-09-30,4.788340e+11,2.452320e+11,2.336020e+11,4.576500e+10,2.977400e+10,1.580630e+11,1.714860e+11,1.592370e+11,2.706200e+10,1.321750e+11,8.98,1.472031e+10


## 3. Data checks (never trust data blindly)

In [6]:
# Check 1 – Accounting equation: Total Assets = Total Liabilities + Equity
check_bs = balance_sheet_clean["Total Liabilities Net Minority Interest"] + balance_sheet_clean["Stockholders Equity"]
print((check_bs - balance_sheet_clean["Total Assets"]))

# Check 2 – Free Cash Flow = Operating Cash Flow + Capex (Capex is already negative)
check_cf = cash_flow_clean["Operating Cash Flow"] + cash_flow_clean["Capital Expenditure"]
print((check_cf - cash_flow_clean["Free Cash Flow"]))

# Both differences should be 0

2023-09-30    0.0
2024-09-30    0.0
2025-09-30    0.0
2026-09-30    0.0
dtype: float64
2023-09-30    0.0
2024-09-30    0.0
2025-09-30    0.0
2026-09-30    0.0
dtype: float64


## 4. Save to CSV

In [7]:
if REFRESH_DATA:
    Path("../data/raw").mkdir(parents=True, exist_ok=True)
    price_long.to_csv("../data/raw/price_data_long.csv", index=False)
    income_statement_clean.to_csv("../data/raw/yf_income_statement.csv")
    balance_sheet_clean.to_csv("../data/raw/yf_balance_sheet.csv")
    cash_flow_clean.to_csv("../data/raw/yf_cash_flow.csv")
    print("Updated raw data; review all dated model inputs before publishing.")
else:
    print("Using bundled raw data; no live download required.")


Using bundled raw data; no live download required.


## 5. Push to MySQL

The 10-K tables in `../data/` (clean names, $ millions) are pushed together with the long price table.
The password is typed in when the cell runs, so it is **never saved in the notebook** (safe to put on GitHub).

In [8]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import getpass

user = "aapl_user"
password = os.environ.pop("MYSQL_PASSWORD", None) or getpass.getpass("MySQL password: ")
engine = create_engine(URL.create("mysql+pymysql", username=user, password=password,
                                 host="127.0.0.1", database="aapl_stock_project"))
with engine.connect():
    print("Connected to MySQL")


Connected to MySQL


In [9]:
import subprocess
import sys

# The loader retains existing tables as backups and validates all seven SQL files.
loader_env = os.environ.copy()
loader_env["MYSQL_PASSWORD"] = password
try:
    result = subprocess.run([sys.executable, "../scripts/validate_mysql.py", "--load"],
                            env=loader_env, capture_output=True, text=True, check=True)
    print(result.stdout)
finally:
    loader_env.pop("MYSQL_PASSWORD", None)
    password = None
    engine.dispose()


01_setup_schema.sql PASS [5, 5]
02_growth_analysis.sql PASS [5]
03_margin_trend.sql PASS [5]
04_liquidity_leverage.sql PASS [5]
05_cash_returns.sql PASS [5]
06_product_mix.sql PASS [25]
07_price_vs_earnings.sql PASS [5, 60]
PASS: all tables match the CSV snapshot; accounting and product checks passed.

